In [6]:
def evaluate(expr, model):
    if isinstance(expr, str):
        return model[expr]
    op = expr[0]
    if op == 'NOT':
        return not evaluate(expr[1], model)
    if op == 'AND':
        return evaluate(expr[1], model) and evaluate(expr[2], model)
    if op == 'OR':
        return evaluate(expr[1], model) or evaluate(expr[2], model)
    if op == 'IMPLIES':
        return (not evaluate(expr[1], model)) or evaluate(expr[2], model)
    if op == 'IFF':
        return evaluate(expr[1], model) == evaluate(expr[2], model)

def tt_check_all(kb, query, symbols, model):
    if not symbols:
        return evaluate(query, model) if evaluate(kb, model) else True
    p = symbols[0]
    rest = symbols[1:]
    return (tt_check_all(kb, query, rest, {**model, p: True}) and
            tt_check_all(kb, query, rest, {**model, p: False}))

def get_symbols(expr):
    if isinstance(expr, str):
        return {expr}
    return set().union(*(get_symbols(sub) for sub in expr[1:]))

def tt_entails(kb, query):
    symbols = list(get_symbols(kb) | get_symbols(query))
    return tt_check_all(kb, query, symbols, {})

s1 = ('IFF', 'P', 'Q')
s2 = ('OR', 'Q', 'R')
s3 = ('IMPLIES', ('NOT', 'P'), 'R')

kb = ('AND', ('AND', s1, s2), s3)

queries = {
    "Q": 'Q',
    "R -> Q": ('IMPLIES', 'R', 'Q'),
    "P -> R": ('IMPLIES', 'P', 'R')
}

for name, q in queries.items():
    print(f"KB entails {name}? -> {tt_entails(kb, q)}")

KB entails Q? -> False
KB entails R -> Q? -> False
KB entails P -> R? -> False
